# انسان در حلقه: دروازه‌های پیش‌عمل، درجه‌بندی ریسک، و ثبت لاگ بررسی

README این درس، انسان در حلقه را با یک قطعه کوتاه معرفی می‌کند که پس از اینکه عامل پاسخ تولید کرده، از کاربر می‌خواهد `APPROVE` یا `REJECT` را انتخاب کند. این الگو نقطه شروع خوبی است، اما پیاده‌سازی‌های تولیدی HITL معمولاً به سه بخش اضافی نیاز دارند:

1. یک **دروازه پیش‌عمل** که **قبل از** اجرای یک گام پرخطر توسط عامل اجرا شود، تا هزینه، برگشت‌ناپذیری و تأخیر تحت کنترل باقی بماند.
2. **درجه‌بندی ریسک**، به طوری که اقدامات کم‌ریسک به صورت خودکار اجرا شوند، اقدامات متوسط ریسک به صورت دسته‌ای تأیید شوند، و فقط اقدامات پرریسک روی انسان توقف کنند.
3. یک **ثبت لاگ بررسی به علاوه حلقه بازبینی**، به طوری که هر تصمیم دروازه به صورت JSONL ثبت شود و رد شدن باعث شود عامل با دلیلی ساختاریافته به جای اینکه فقط `Revising...` چاپ کند، مجدداً درخواست شود.

این دفترچه یادداشت هرکدام از این موارد را روی همان اجزای اصلی `06-system-message-framework.ipynb` می‌سازد. این کد به صورت انتها به انتها در `DEMO_MODE = True` (نیازی به ورودی تعاملی ندارد) یا با پرسش واقعی `input()` وقتی که `DEMO_MODE = False` است، اجرا می‌شود. نکته: در DEMO_MODE تلاش مجدد هدف سوم طوری اسکریپت شده است که مکانیک حلقه به صورت کامل قابل مشاهده باشد. بازطبقه‌بندی مبتنی بر بازبینی واقعی نیازمند `DEMO_MODE = False` و یک اپراتور است.

**خارج از محدوده (در درس‌های دیگر پوشش داده شده):** احراز هویت و کنترل دسترسی (تهدید 2 در README درس 06)، میان‌افزار فراخوانی ابزار (بررسی عمیق MAF در درس 14)، الگوهای مناظره چندعامله.

In [ ]:
import json
import os
from datetime import datetime, timezone
from pathlib import Path

from dotenv import load_dotenv
from azure.ai.inference import ChatCompletionsClient
from azure.ai.inference.models import SystemMessage, UserMessage
from azure.core.credentials import AzureKeyCredential

load_dotenv()

DEMO_MODE = True  # set False to use real input() prompts

# Per-run unique log filename so demo runs don't overwrite each other and
# the notebook doesn't touch any pre-existing gate_log.jsonl in the working
# directory.
GATE_LOG_PATH = Path(
    f"gate_log_{datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')}.jsonl"
)

token = os.environ.get("GITHUB_TOKEN", "")
if not token:
    raise RuntimeError(
        "GITHUB_TOKEN environment variable is not set. This notebook needs "
        "a GitHub PAT with 'Models: Read-only' permission. Set GITHUB_TOKEN "
        "in your environment or a local .env file before running."
    )

endpoint = "https://models.github.ai/inference"
model_name = "gpt-4o"

client = ChatCompletionsClient(
    endpoint=endpoint,
    credential=AzureKeyCredential(token),
)


## الگو ۱: دروازه قبل از عمل

کد نمونه HITL در README ابتدا عامل را فرا می‌خواند، سپس از کاربر می‌خواهد خروجی را تأیید کند. این یک جریان **پس از عمل** است. عامل قبلاً اجرا شده است، بنابراین هزینه فراخوانی LLM قبلاً پرداخت شده و هر تأثیر جانبی (ارسال ایمیل، نوشتن ردیف در پایگاه داده، ارسال نظر) قبلاً رخ داده است.

جریان **قبل از عمل** دروازه را قبل از اجرای گام پرخطر عامل قرار می‌دهد. عامل عمل را پیشنهاد می‌دهد، دروازه تصمیم می‌گیرد که آیا اجرا شود، و فقط در صورت تأیید، تأثیر جانبی رخ می‌دهد.

| جنبه | تأیید پس از عمل (کد نمونه README) | دروازه قبل از عمل (این دفترچه) |
|---|---|---|
| تأیید کی اجرا می‌شود؟ | پس از اینکه عامل خروجی تولید کرد | قبل از اجرای هر تأثیر جانبی |
| هزینه LLM در صورت رد | قبلاً پرداخت شده | فقط برای پیشنهاد پرداخت می‌شود، نه عمل |
| تأثیرات جبران‌ناپذیر | ممکن است (عمل قبلاً انجام شده است) | جلوگیری شده است |
| وضوح حسابرسی | تأیید یک عبارت چاپی است | تأیید یک رکورد JSON با زمان، عمل، دلیل است |


In [ ]:
def gate_action(action_description: str, risk_tier: str, attempt: int = 0) -> dict:
    """Run a single pre-action gate.

    Returns a decision dict with keys: decision, reason, ts.
    Decision is one of: approve, deny, escalate.
    Safe default on EOF or unexpected input is deny.

    DEMO_MODE behavior: high-risk actions are denied on attempt 0 and
    auto-approved on attempt >= 1. This is scripted approval to show the
    loop mechanics (deny -> retry -> approve). It is NOT revision-driven
    re-classification. Real revision-driven re-classification requires
    DEMO_MODE=False and a human operator who evaluates the revised
    proposal on its own merits.
    """
    print(f"[gate] proposed action ({risk_tier}, attempt={attempt}): {action_description}")

    if DEMO_MODE:
        if risk_tier == "high":
            decision = "approve" if attempt >= 1 else "deny"
            reason = (
                "DEMO_MODE: scripted approval on retry to show loop mechanics"
                if attempt >= 1
                else "DEMO_MODE: high risk denied on first attempt"
            )
        else:
            decision = "approve"
            reason = f"DEMO_MODE canned response for tier={risk_tier}"
    else:
        try:
            raw = input("[gate] approve / deny / escalate? ").strip().lower()
        except EOFError:
            raw = ""
        if raw in {"approve", "deny", "escalate"}:
            decision, reason = raw, "operator input"
        elif raw == "":
            decision, reason = "deny", "no input received, defaulted to deny"
        else:
            decision, reason = "deny", f"invalid input {raw!r}, defaulted to deny"

    return {
        "decision": decision,
        "reason": reason,
        "action": action_description,
        "risk_tier": risk_tier,
        "ts": datetime.now(timezone.utc).isoformat(),
    }


## الگو ۲: دسته‌بندی ریسک

نیاز نیست هر عملی توسط انسان تأیید شود. جستجوی فقط خواندنی در برابر یک API عمومی نسبت به ارسال ایمیل به مشتری اهمیت متفاوتی دارد. برخورد یکسان با هر دو باعث هدر رفتن توجه اپراتور و کند شدن عامل می‌شود.

یک مدل ساده ۳ سطحی:

| سطح | مثال‌ها | جریان تأیید |
|---|---|---|
| `کم` (فقط خواندنی) | جستجوی پایگاه دانش، جستجوی گزینه‌های پرواز، دریافت یک صفحه وب عمومی | اجرا خودکار، ثبت برای حسابرسی |
| `متوسط` (تغییر ارزان) | ذخیره نتیجه، افزایش شمارنده، برنامه‌ریزی یادآوری | اجرا خودکار، اما بازبینی جمعی روزانه |
| `زیاد` (مواجهه خارجی یا غیرقابل بازگشت) | ارسال ایمیل، شارژ کارت، ارسال به کانال عمومی | مسدود شدن تا تأیید انسانی |

این یک دسته‌بندی است. سیستم‌های تولیدی معمولاً از دسته‌بندی‌های دقیق‌تر استفاده می‌کنند (مثلاً سطوح مجوز AWS IAM، دسته‌های دسترسی مبتنی بر نقش). نسخه ۳ سطحی زیر کوچک‌ترین نسخه مفید برای عاملی است که کارهای فقط خواندنی و با اثر جانبی را ترکیب می‌کند.

رده‌بند زیر از قواعد کلمه کلیدی استفاده می‌کند تا نسخه نمایشی قطعی و ارزان باقی بماند. در یک سیستم تولیدی، شما این را با یک رده‌بند یادگرفته شده یا موتور سیاست جایگزین می‌کنید.

In [ ]:
LOW_RISK_KEYWORDS = {
    "look", "lookup", "search", "fetch", "read", "query", "view",
    "get", "list", "weather", "summarize",
}
HIGH_RISK_KEYWORDS = {
    "send", "email", "post", "publish", "charge", "pay", "transfer",
    "delete", "drop", "cancel", "refund",
}
MEDIUM_RISK_KEYWORDS = {
    "cache", "schedule", "reminder", "book", "reserve", "update",
    "increment", "log",
}

AUTO_APPROVE_REASONS = {
    "low": "auto-approved (low risk)",
    "medium": "auto-approved (medium risk, queued for batched review)",
}


def classify_risk(action: str) -> str:
    """Classify an action string into one of: low, medium, high.

    Keyword-based heuristic. Checks high-risk first (most severe), then
    low-risk explicit reads, then medium-risk mutations. Unrecognized
    actions default to medium, not low.

    Default for unrecognized actions is 'medium', not 'low'. A read-only
    keyword set will always have blind spots, and the parent README's
    threat list (critical-system access, knowledge-base poisoning,
    cascading errors) all involve cases an action-name alone cannot rule
    out. Routing unknown actions through batched review is the safer
    default than auto-executing them.
    """
    text = action.lower()
    if any(kw in text for kw in HIGH_RISK_KEYWORDS):
        return "high"
    if any(kw in text for kw in LOW_RISK_KEYWORDS):
        return "low"
    if any(kw in text for kw in MEDIUM_RISK_KEYWORDS):
        return "medium"
    # Explicit fail-safe default: unrecognized actions route to batched review.
    return "medium"


def tiered_gate(action: str, attempt: int = 0) -> dict:
    """Classify then gate. Low and medium tiers auto-approve; high blocks."""
    tier = classify_risk(action)
    if tier in AUTO_APPROVE_REASONS:
        return {
            "decision": "approve",
            "reason": AUTO_APPROVE_REASONS[tier],
            "action": action,
            "risk_tier": tier,
            "ts": datetime.now(timezone.utc).isoformat(),
        }
    return gate_action(action, tier, attempt=attempt)


## الگوی ۳: گزارش ممیزی و حلقه بازبینی

یک `print("Response approved.")` گزارش ممیزی نیست. برای ایجاد اعتماد، هر تصمیم گیت باید به عنوان یک رویداد ساختاریافته ثبت شود که بعدها بتوانید آن را جستجو، بازپخش یا به بررسی حادثه ضمیمه کنید.

دو بخش:

۱. **فایل JSONL افزودنی فقط به انتها.** یک خط برای هر تصمیم، با زمان، عمل، سطح، تصمیم، دلیل. آسان برای جستجو و آسان برای ارسال به یک مخزن گزارش واقعی بعداً.  
۲. **حلقه بازبینی هنگام رد شدن.** وقتی گیت `deny` برمی‌گرداند، عامل خودش را با دلیل رد شدن در متن مجدداً درخواست می‌کند تا پیشنهاد بعدی بتواند از مشکل جلوگیری کند.

In [ ]:
def log_decision(decision: dict) -> None:
    """Append a gate decision to the JSONL audit log."""
    with GATE_LOG_PATH.open("a", encoding="utf-8") as f:
        f.write(json.dumps(decision) + "\n")


def propose_action(goal: str, prior_rejection: str | None = None) -> str:
    """Ask the LLM to propose a concrete next action for a goal.

    If prior_rejection is provided, it is fed back so the LLM can avoid
    the same failure mode in the next proposal.
    """
    system = (
        "You are an action planner for an agent. Propose ONE concrete next\n"
        "action (a single sentence) toward the user's goal. If a prior\n"
        "rejection reason is given, propose a different action that addresses\n"
        "the rejection."
    )
    user_text = f"Goal: {goal}"
    if prior_rejection:
        user_text += f"\n\nPrior proposal was denied. Reason: {prior_rejection}"

    response = client.complete(
        model=model_name,
        messages=[SystemMessage(content=system), UserMessage(content=user_text)],
    )
    return response.choices[0].message.content.strip()


def run_with_revision(goal: str, max_revisions: int = 2) -> dict:
    """Propose, gate, and on rejection revise up to max_revisions times."""
    prior_reason: str | None = None
    for attempt in range(max_revisions + 1):
        action = propose_action(goal, prior_rejection=prior_reason)
        decision = tiered_gate(action, attempt=attempt)
        decision["attempt"] = attempt
        log_decision(decision)
        if decision["decision"] == "approve":
            return decision
        prior_reason = decision["reason"]
    return {**decision, "final": "max_revisions_reached"}


In [ ]:
# End-to-end demo: three goals at three different risk profiles.
# GATE_LOG_PATH is per-run (timestamped) so no prior log is touched.

goals = [
    "Look up the weather in Seattle for the customer's trip planning.",
    "Schedule a reminder for the customer to check in 24 hours before their flight.",
    "Send a marketing email to the customer about premium upgrade options.",
]

for goal in goals:
    print(f"\n=== Goal: {goal} ===")
    outcome = run_with_revision(goal, max_revisions=1)
    print(f"[final] {outcome['decision']} ({outcome['reason']})")

print(f"\n=== Audit log ({GATE_LOG_PATH.name}) ===")
for line in GATE_LOG_PATH.read_text(encoding="utf-8").splitlines():
    record = json.loads(line)
    print(f"  [{record['risk_tier']:6s}] {record['decision']:8s} "
          f"attempt={record.get('attempt', '?')} action={record['action'][:140]}")


## منابع اضافی

چند پروژه عمومی دیگر انواع مختلفی از این الگوهای HITL را پیاده‌سازی می‌کنند. روش‌ها را مقایسه کنید تا ببینید چه چیزی برای پشته شما مناسب است:

- **LangChain** بسته‌های ابزاری انسان در حلقه ([مستندات](https://python.langchain.com/docs/integrations/tools/human_tools)): بسته‌های ابزاری که اجرای برنامه را برای ورودی انسانی متوقف می‌کنند.
- **AutoGen** `UserProxyAgent` ([مستندات v0.2](https://microsoft.github.io/autogen/0.2/docs/topics/human-in-the-loop); AutoGen v0.4+ این مورد را بازساخت کرد): نقشی از نماینده را به طور خاص برای نمایندگی انسان در گفتگوهای چند نماینده استفاده می‌کند.
- **Semantic Kernel** فیلترهای تابع ([مستندات](https://learn.microsoft.com/en-us/semantic-kernel/concepts/enterprise-readiness/filters)): نرم‌افزاری میانی که در اطراف هر فراخوان تابع اجرا می‌شود، مناسب برای منطق کنترل دسترسی.

هر پروژه سه زیرالگوی مختلف را به روشی متفاوت مدیریت می‌کند: LangChain آنها را به عنوان ابزار بسته‌بندی می‌کند، AutoGen از نقش نماینده استفاده می‌کند، Semantic Kernel از فیلترهای میانی استفاده می‌کند. یک یا دو پیاده‌سازی را از ابتدا تا انتها بخوانید قبل از اینکه طراحی نماینده خود را انتخاب کنید.

---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**سلب مسئولیت**:
این سند با استفاده از سرویس ترجمه هوش مصنوعی [Co-op Translator](https://github.com/Azure/co-op-translator) ترجمه شده است. در حالی که ما در تلاش برای دقت هستیم، لطفاً توجه داشته باشید که ترجمه‌های خودکار ممکن است شامل خطاها یا نادرستی‌هایی باشند. سند اصلی به زبان مادری خود باید به عنوان منبع معتبر در نظر گرفته شود. برای اطلاعات حیاتی، ترجمه حرفه‌ای انسانی توصیه می‌شود. ما در قبال هرگونه سوء تفاهم یا برداشت نادرست ناشی از استفاده از این ترجمه مسئولیتی نداریم.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
